In [1]:
!pip3 install langchain langchain-community langchain-experimental langchain-huggingface langchain-ollama sentence-transformers faiss-cpu

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_experimental-0.4.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached langchain_ollama-1.1.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached faiss_cpu-1.15.0-cp314-cp314-win_amd64.whl.metadata (7.8 kB)
  Using cached langchain_core-1.6.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using ca


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from pathlib import Path
import pandas as pd

from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama


# =====================================================
# Configuration
# =====================================================

ARTICLES_DIR = Path("AI_Knowledge_Base")
METADATA_FILE = ARTICLES_DIR / "metadata.csv"

In [9]:
def load_text_documents():
    documents = []

    metadata_df = pd.read_csv(METADATA_FILE)

    metadata_lookup = {
        row["File"]: row
        for _, row in metadata_df.iterrows()
    }

    for file_path in ARTICLES_DIR.glob("*.txt"):
        text = file_path.read_text(encoding="utf-8").strip()

        if not text:
            continue

        article_metadata = metadata_lookup.get(file_path.name, {})

        documents.append(
            Document(
                page_content=text,
                metadata={
                    "title": article_metadata.get("Title", file_path.stem),
                    "url": article_metadata.get("URL", ""),
                    "keyword": article_metadata.get("Keyword", ""),
                    "source": str(file_path)
                }
            )
        )

    return documents


documents = load_text_documents()

print(f"Loaded {len(documents)} documents")

if documents:
    print(f"\nFirst document title: {documents[0].metadata['title']}")
    print(f"First document text length: {len(documents[0].page_content)} characters")

Loaded 89 documents

First document title: A.I. Artificial Intelligence
First document text length: 26163 characters


In [10]:
print(f"Loaded {len(documents)} documents")

if documents:
    print(f"\nFirst document title: {documents[0].metadata['title']}")
    print(f"First document text length: {len(documents[0].page_content)} characters")

Loaded 89 documents

First document title: A.I. Artificial Intelligence
First document text length: 26163 characters


In [11]:
print(f"Total documents: {len(documents)}")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

splitter = SemanticChunker(embeddings)

print("Creating semantic chunks...")

semantic_docs = splitter.split_documents(documents)

print(f"Total semantic chunks: {len(semantic_docs)}")

if semantic_docs:
    print(f"\nFirst chunk length: {len(semantic_docs[0].page_content)} characters")
    print(f"First chunk source: {semantic_docs[0].metadata['title']}")

Total documents: 89


c:\Users\laksh\Documents\dynamic-wikipedia-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1304.81it/s]


Creating semantic chunks...
Total semantic chunks: 845

First chunk length: 7234 characters
First chunk source: A.I. Artificial Intelligence


In [12]:
print("Building FAISS index...")

vector_db = FAISS.from_documents(
    semantic_docs,
    embeddings
)

vector_db.save_local("faiss-db")

print("FAISS database saved to 'faiss-db' folder")

Building FAISS index...
FAISS database saved to 'faiss-db' folder


In [13]:
query = "What is machine learning?"

print(f"Query: {query}\n")

results = vector_db.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"Result {i}:")
    print(f"  Source: {doc.metadata['title']}")
    print(f"  URL: {doc.metadata['url']}")
    print(f"  Text preview: {doc.page_content[:200]}...\n")


Query: What is machine learning?

Result 1:
  Source: Machine learning
  URL: https://en.wikipedia.org/wiki/Machine_learning
  Text preview: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thu...

Result 2:
  Source: Artificial intelligence
  URL: 
  Text preview: The policy could be calculated (e.g., by policy iteration), determined by a heuristic, or learned. Game theory describes the rational behaviour of multiple interacting agents and is used in AI program...

Result 3:
  Source: Artificial intelligence
  URL: 
  Text preview: They can be fine-tuned based on chosen examples using supervised learning. Each pattern (also called an "observation") is labeled with a certain predefined class. All the observations combined with th...



In [14]:
!pip3 install ollama


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from ollama import chat

response = chat(
    model='llama3.1',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)

print(response.message.content)

ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [17]:
from ollama import chat

response = chat(
    model='llama3.1:8b',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)

print(response.message.content)

Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?


In [18]:
query = "What is machine learning?"

print(f"Query: {query}\n")

# Retrieve relevant chunks
results = vector_db.similarity_search(query, k=3)

# Build context from retrieved chunks
context = "\n\n".join(
    f"Source: {doc.metadata['title']}\n{doc.page_content}"
    for doc in results
)

# Create the prompt
prompt = f"""
You are an educational AI assistant.

Answer the question using only the following context.
If the answer is not available in the context, say:
"I could not find the answer in the knowledge base."

Do not invent information.
Give a clear explanation suitable for a beginner.

Context:
{context}

Question:
{query}

Answer:
"""

# Generate the answer
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)

response = llm.invoke(prompt)

print("Answer:")
print(response.content)

print("\nSources:")

shown_sources = set()

for doc in results:
    title = doc.metadata["title"]
    url = doc.metadata["url"]

    if title not in shown_sources:
        print(f"- {title}: {url}")
        shown_sources.add(title)

Query: What is machine learning?

Answer:
According to the context, machine learning is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.

Sources:
- Machine learning: https://en.wikipedia.org/wiki/Machine_learning
- Artificial intelligence: 
